In [189]:
###NOT COMPLETE!!! DO NOT USE UNLESS YOU REALLY KNOW WHAT DA HECK u DOIN'!!!1

import gymnasium as gym
import numpy as np
import torch
from torch import nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from tqdm import tqdm
from torch.optim import Adam

In [220]:
network = nn.Sequential(*[nn.Linear(8, 30), 
                          nn.ReLU(),
                          nn.Linear(30, 40),
                          nn.ReLU(),
                          nn.Linear(40, 4),
                          nn.Softmax(-1),
                         ]); ### This dude is the policy function
network_old = nn.Sequential(*[nn.Linear(8, 30), 
                          nn.ReLU(),
                          nn.Linear(30, 40),
                          nn.ReLU(),
                          nn.Linear(40, 4),
                          nn.Softmax(-1),
                         ]); ### This dude is the policy function
state_value = nn.Sequential(*[nn.Linear(8, 40), 
                          nn.GELU(),
                          nn.Linear(40, 30),
                          nn.GELU(),
                          nn.Linear(30, 1)
                             ]) ### This dude is the state value
#network_old.load_state_dict(network.state_dict())

In [221]:
x = torch.randn(10, 8)
y_pred = network(x)
y_true = network_old(x)
(y_true*(y_true/y_pred).log()).sum()
F.kl_div(y_pred.log(), y_true, reduce = "batchmean")

tensor(0.0026, grad_fn=<MeanBackward0>)

In [229]:
@torch.no_grad
def choose_action(state:np.ndarray, network:nn.Module = network)->int:
    probs = network(torch.tensor(state))
    probs = Categorical(probs).sample()
    return probs.item()

@torch.no_grad
def update(Q, Q_old, α = 0.1):
    for param_Q, param_Q_old in zip(Q.parameters(), Q_old.parameters()):
        param_Q_old += α*(param_Q-param_Q_old)       

In [236]:
env = gym.make("LunarLander-v2", 
               gravity = -10.0,
               enable_wind = True,
    wind_power = 2.0)
γ = 0.99
β = 0.001

opt1 = Adam(network.parameters(), 0.001)
opt2 = Adam(state_value.parameters(), 0.001)

num_iters = tqdm(range(1500))

for i in num_iters:
    last_state, info = env.reset()
    I = 1
    cum_reward = 0
    cum_delta = 0

    network.train()
    state_value.train()
    
    opt1.zero_grad()
    opt2.zero_grad()
    
    z = torch.tensor([0.0], requires_grad = True)
    
    z_ = torch.tensor([0.0], requires_grad = True)
    for c in range(4000):
        
        action = choose_action(last_state, network_old)  # agent policy that uses the observation and info
        current_state, reward, terminated, truncated, info = env.step(action)

        with torch.no_grad():

            if terminated or truncated:
                δ = reward - state_value(torch.tensor(last_state))
            else:
                δ = reward + γ*state_value(torch.tensor(current_state)) - state_value(torch.tensor(last_state))
        #Kalpazanlar andreaj jeet               
        cum_reward = (0.9)*cum_reward+(0.1)*(reward)
        cum_delta = (0.9)*cum_delta+(0.1)*(δ)
                
        
        z = z.add(-δ*state_value(torch.tensor(last_state)))
        
        with torch.no_grad():
            old_choice = network_old(torch.tensor(last_state)).squeeze()

        z_ = z_.add(-δ*I*(network(torch.tensor(last_state)).squeeze()[action]/old_choice[action]))
        #similarity = F.kl_div(network(torch.tensor(last_state)).log(), old_choice, reduction = "batchmean")
        #z_ = z_.add(β*(similarity))
        I *= γ
        if terminated or truncated:
            break
        else:
            last_state = current_state
    #
    num_iters.set_description(f"{cum_reward, z_.item(), cum_delta.item()}")

    z.backward() ##Update state val
    z_.backward() ## Update the dude!!!
  
    opt1.step()
    opt2.step()
    
    update(network_old, network)
    
        
    
env.close()       
    #update_params(network, state_value,states, actions, rewards)
    #update_params(network, state_value,states, actions, rewards)

(-5.01015516195668, -131.09861755371094, 4.9679365158081055): 100%|████████████████| 1500/1500 [03:25<00:00,  7.30it/s]


In [235]:
env = gym.make("LunarLander-v2", 
               max_episode_steps = 700,  
               gravity = -10.0,
               enable_wind = False,
               wind_power =2.0,
               render_mode = "human"
              )

state, info = env.reset()
γ = 0.99

for c in range(700):
    network.eval()
    action = choose_action(state, network)  # agent policy that uses the observation and info
    state, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        state, info = env.reset()
        break
env.close()